In [1]:
import json

# Define the manifest structure
manifest_data = {
    "manifest_version": 3,
    "name": "Fake Review Radar",
    "version": "1.0.0",
    "description": "Rates reviews as Non-AI, Uncertain, Likely AI, or Surely AI.",
    "permissions": ["storage", "activeTab", "scripting"],
    "host_permissions": ["http://localhost:8080/*"],
    "background": {
        "service_worker": "background.js"
    },
    "content_scripts": [
        {
            "matches": ["*://*.amazon.com/*", "*://*.yelp.com/*", "*://*/*"],
            "js": ["contentScript.js"],
            "run_at": "document_idle"
        }
    ],
    "action": {
        "default_popup": "popup.html",
        "default_title": "Fake Review Radar"
    }
}

# Save the dictionary as a manifest.json file
with open('manifest.json', 'w') as f:
    json.dump(manifest_data, f, indent=4)

# Optional: Create empty placeholder files so Chrome doesn't throw "file not found" errors
for filename in ['background.js', 'contentScript.js', 'popup.html']:
    with open(filename, 'w') as f:
        f.write(f'// Placeholder for {filename}')

print("Manifest and placeholder files created successfully!")


Manifest and placeholder files created successfully!


In [2]:
# Define the background script content
background_js_content = """
const DEFAULT_API_URL = "http://localhost:8080/v1/review/batch";

chrome.runtime.onInstalled.addListener(() => {
  chrome.storage.sync.set({
    apiUrl: DEFAULT_API_URL,
    apiKey: "local-dev-key"
  });
});

chrome.runtime.onMessage.addListener((message, sender, sendResponse) => {
  if (message.type !== "SCORE_REVIEWS") return;

  chrome.storage.sync.get(["apiUrl", "apiKey"], ({ apiUrl, apiKey }) => {
    fetch(apiUrl, {
      method: "POST",
      headers: {
        "Content-Type": "application/json",
        "X-API-Key": apiKey || ""
      },
      body: JSON.stringify({ reviews: message.payload.reviews })
    })
      .then((response) => response.json())
      .then((data) => sendResponse({ ok: true, data }))
      .catch((error) => sendResponse({ ok: false, error: error.toString() }));
  });

  return true; // keep message channel open for async response
});
"""

# Save the string into background.js
with open('background.js', 'w') as f:
    f.write(background_js_content.strip())

print("background.js has been created in your working directory.")


background.js has been created in your working directory.


In [3]:
# Define the content script logic
content_script_js = """
const REVIEW_SELECTORS =",
  "div.review"
];

const BADGE_CLASS = "fake-review-radar-badge";

function collectReviewNodes() {
  const nodes = new Set();
  REVIEW_SELECTORS.forEach((selector) => {
    document.querySelectorAll(selector).forEach((node) => {
      const text = node.innerText?.trim() || "";
      // Filter out short snippets to avoid noise
      if (text.length >= 40) nodes.add(node);
    });
  });
  return Array.from(nodes);
}

function injectBadge(targetNode, verdict) {
  // Prevent duplicate badges if the script runs multiple times
  const existing = targetNode.querySelector(`.${BADGE_CLASS}`);
  if (existing) existing.remove();

  const badge = document.createElement("div");
  badge.className = BADGE_CLASS;
  badge.textContent = `AI Check: ${verdict.bucket}`;
  badge.title = `${verdict.explanation}\\nAI probability: ${(verdict.prob_fake * 100).toFixed(1)}%`;
  
  // Basic styling for the badge
  badge.style.cssText = `
    background: #1d3557;
    color: #f1faee;
    font-size: 12px;
    font-weight: 600;
    display: inline-block;
    padding: 4px 8px;
    border-radius: 8px;
    margin-bottom: 6px;
    font-family: sans-serif;
  `;

  targetNode.prepend(badge);
}

function annotateReviews(results) {
  const nodes = collectReviewNodes();
  // Match the API results back to the DOM nodes
  results.slice(0, nodes.length).forEach((verdict, idx) => {
    injectBadge(nodes[idx], verdict);
  });
}

function scoreReviewsOnPage() {
  const nodes = collectReviewNodes();
  if (!nodes.length) return;

  const reviews = nodes.map((node) => node.innerText.trim());
  
  // Send reviews to background.js for API processing
  chrome.runtime.sendMessage(
    { type: "SCORE_REVIEWS", payload: { reviews } },
    (response) => {
      if (response?.ok) {
          annotateReviews(response.data.results);
      } else {
          console.warn("Fake Review Radar error:", response?.error);
      }
    }
  );
}

// Watch for dynamic content loading (like "Load More" buttons or infinite scroll)
const observer = new MutationObserver(() => {
    // Using a simple debounce could be added here to improve performance
    scoreReviewsOnPage();
});

observer.observe(document.body, { childList: true, subtree: true });

// Initial run when the page finishes loading
window.addEventListener("load", () => scoreReviewsOnPage());
"""

# Save the string into contentScript.js
with open('contentScript.js', 'w') as f:
    f.write(content_script_js.strip())

print("contentScript.js has been created!")


contentScript.js has been created!


In [4]:
# Define the popup UI content
popup_html_content = """
<!DOCTYPE html>
<html>
  <head>
    <meta charset="utf-8" />
    <title>Fake Review Radar Settings</title>
    <style>
      body {
        font-family: "Inter", system-ui, sans-serif;
        width: 260px;
        margin: 0;
        padding: 12px;
        background: #f8f9fc;
      }
      h3 {
        margin: 0 0 16px 0;
        color: #1d3557;
      }
      label {
        display: block;
        margin-top: 10px;
        font-weight: 600;
        font-size: 13px;
        color: #475569;
      }
      input {
        width: 100%;
        padding: 6px;
        margin-top: 4px;
        border-radius: 6px;
        border: 1px solid #cbd5e1;
        box-sizing: border-box;
      }
      button {
        margin-top: 14px;
        padding: 8px 12px;
        width: 100%;
        background: #1d3557;
        color: white;
        border: none;
        border-radius: 6px;
        font-weight: 600;
        cursor: pointer;
      }
      button:hover {
        background: #457b9d;
      }
    </style>
  </head>
  <body>
    <h3>Fake Review Radar</h3>
    <label for="apiUrl">API URL</label>
    <input id="apiUrl" placeholder="http://localhost:8080/v1/review/batch" />
    <label for="apiKey">API Key</label>
    <input id="apiKey" type="password" placeholder="local-dev-key" />
    <button id="saveBtn">Save Settings</button>
    <script src="popup.js"></script>
  </body>
</html>
"""

# Save the string into popup.html
with open('popup.html', 'w') as f:
    f.write(popup_html_content.strip())

print("popup.html has been created!")


popup.html has been created!


In [6]:
# Define the popup script logic
popup_js_content = """
const apiUrlInput = document.getElementById("apiUrl");
const apiKeyInput = document.getElementById("apiKey");
const saveBtn = document.getElementById("saveBtn");

chrome.storage.sync.get(["apiUrl", "apiKey"], ({ apiUrl, apiKey }) => {
  apiUrlInput.value = apiUrl || "";
  apiKeyInput.value = apiKey || "";
});

saveBtn.addEventListener("click", () => {
  chrome.storage.sync.set(
    {
      apiUrl: apiUrlInput.value.trim(),
      apiKey: apiKeyInput.value.trim()
    },
    () => {
      saveBtn.textContent = "Saved ✅";
      setTimeout(() => (saveBtn.textContent = "Save Settings"), 1500);
    }
  );
});
"""

# FIX: Added encoding='utf-8' to handle the checkmark emoji
with open('popup.js', 'w', encoding='utf-8') as f:
    f.write(popup_js_content.strip())

print("popup.js has been created successfully with UTF-8 encoding!")


popup.js has been created successfully with UTF-8 encoding!


In [7]:
# Define the README content
readme_content = """
# Fake Review Radar Extension

1. Open Chrome or Edge → `chrome://extensions/`
2. Toggle **Developer mode** on.
3. Choose **Load unpacked** and select this `extension/` folder.
4. Click the extension icon → configure API URL & API key (default in dev: `http://localhost:8080/v1/review/batch`, `local-dev-key`).
5. Browse review-heavy pages (Amazon, Yelp, etc.). Badges appear automatically with the AI verdicts.

> Pair this with your Flask API or whichever endpoint your Gradio app will eventually call. When the real model is in place, predictions flow straight into the per-review badges via the `prob_fake` and `bucket` fields returned by the backend.
"""

# Save with utf-8 encoding to handle the arrow and emoji characters
with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content.strip())

print("README.md has been created successfully!")


README.md has been created successfully!


In [9]:
import json

# Example response structure for the AI model
response_data = {
  "results": [
    {
      "bucket": "Likely AI",
      "prob_fake": 0.72,
      "explanation": "High repetition and generic praise typical of Large Language Models.",
      "text": "The original review text analyzed goes here..."
    }
  ],
  "count": 1
}

# Save as a reference file
with open('sample_response.json', 'w', encoding='utf-8') as f:
    json.dump(response_data, f, indent=4)

print("sample_response.json created! Ensure your backend sends this exact structure.")


sample_response.json created! Ensure your backend sends this exact structure.
